# 13 — Partitioning & Shuffle Optimization

**What you will learn:**
- What a Spark partition is and why size matters
- `repartition()` vs `coalesce()` — when to use each
- `spark.sql.shuffle.partitions` — tuning shuffle output
- Data skew: detecting and fixing with salting
- `partitionBy()` on write — file-level partitioning for reads
- `bucketBy()` — eliminate join shuffles permanently

## 1. What is a Spark Partition?

A partition is a **chunk of data** processed by a single task on one executor.

```
DataFrame
  Partition 0  →  Task 0  →  Executor 1
  Partition 1  →  Task 1  →  Executor 2
  Partition 2  →  Task 2  →  Executor 1
```

- Too **few** partitions → huge tasks, executors sit idle
- Too **many** partitions → scheduling overhead, tiny files on write
- **Skewed** partitions → one task 10× longer than others (straggler)

**Rule of thumb:** aim for 128 MB – 256 MB per partition after a shuffle.

In [ ]:
import pyspark.sql.functions as F

df = spark.createDataFrame(
    [(i, f"emp_{i}", ["Engineering","Marketing","HR","Finance"][i%4],
      50000.0 + (i * 137 % 70000)) for i in range(1, 300001)],
    ["id","name","department","salary"]
)
print(f"Rows:            {df.count():,}")
print(f"Partitions:      {df.rdd.getNumPartitions()}")

## 2. repartition() — Increase or Redistribute Partitions

- Triggers a **full shuffle** — data redistributed across cluster
- Use when you need **more parallelism** or want to **balance skewed partitions**
- `repartition("col")` — hash partitions by column value, co-locating related rows

In [ ]:
# Increase partition count (full shuffle)
df_200 = df.repartition(200)
print("repartition(200):         ", df_200.rdd.getNumPartitions())

# Hash-partition by column (rows with same dept → same partition)
df_by_dept = df.repartition("department")
print("repartition('department'):", df_by_dept.rdd.getNumPartitions())

# Explicit count + column
df_4 = df.repartition(4, "department")
print("repartition(4,'department'):", df_4.rdd.getNumPartitions())

## 3. coalesce() — Reduce Partitions Without Full Shuffle

- Merges partitions **locally** — no full shuffle
- Can only **reduce** (not increase) partition count
- Perfect before writing small results — avoids many tiny output files

In [ ]:
df_small = df.filter(F.col("department") == "Finance").coalesce(2)
print("coalesce(2):", df_small.rdd.getNumPartitions())

# Decision guide:
# repartition(n)    — increase OR decrease; full shuffle; balanced output
# coalesce(n)       — decrease only; no shuffle; possibly uneven sizes
# repartition(col)  — hash-partition before a join on that column

## 4. spark.sql.shuffle.partitions

Every shuffle (groupBy, join, orderBy) creates output files.
Count is controlled by `spark.sql.shuffle.partitions` (default **200**).

- 200 on small data → 200 tiny files, huge scheduler overhead
- 200 on large data → partitions may be too large → OOM

**Formula:** `shuffle_partitions ≈ shuffle_data_GB × 1024 / target_MB_per_partition`

In [ ]:
print("Default:", spark.conf.get("spark.sql.shuffle.partitions"))

# For development on small data — reduce to 8
spark.conf.set("spark.sql.shuffle.partitions", "8")

result = df.groupBy("department").agg(F.avg("salary"), F.count("id"))
result.show()
print("Result partitions:", result.rdd.getNumPartitions())

# For production with AQE enabled — set high, AQE coalesces down automatically
# spark.conf.set("spark.sql.shuffle.partitions", "800")

## 5. Data Skew — Detection & Salting Fix

**Skew** = one partition has far more data than others → one task is the straggler.

**Detection:** Spark UI → Stages tab → look for one task with much higher duration than the rest.

**Fix: Salting** — add a random key to break the skewed partition into sub-partitions.

In [ ]:
# Simulate skewed data: 80% rows have department = "Engineering"
skew = (
    [(i, "Engineering", 90000.0) for i in range(1, 80001)] +
    [(i, "Marketing",   70000.0) for i in range(80001, 90001)] +
    [(i, "HR",          65000.0) for i in range(90001, 100001)]
)
df_skewed = spark.createDataFrame(skew, ["id","department","salary"])
df_skewed.groupBy("department").count().show()
# Engineering will be a straggler partition in any join/groupBy

In [ ]:
SALT_BUCKETS = 4

# Step 1: add a random salt column
df_salted = df_skewed.withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))

# Step 2: first aggregation on (department + salt) — load balanced
df_partial = df_salted.groupBy("department","salt").agg(
    F.count("id").alias("partial_count"),
    F.sum("salary").alias("partial_sum")
)

# Step 3: second aggregation removes salt — final result
df_final = df_partial.groupBy("department").agg(
    F.sum("partial_count").alias("total_count"),
    F.sum("partial_sum").alias("total_salary")
)
df_final.show()

## 6. partitionBy() on Write — File-Level Partitioning

Writing with `partitionBy` organizes files into sub-folders by column value.
Queries filtering on that column **skip irrelevant folders** — partition pruning.

In [ ]:
(
    df.write.format("delta").mode("overwrite")
      .partitionBy("department")
      .save("/tmp/opt/emp_partitioned/")
)

# Only the Engineering folder is read — massive I/O reduction
df_pruned = (
    spark.read.format("delta")
         .load("/tmp/opt/emp_partitioned/")
         .filter(F.col("department") == "Engineering")
)
df_pruned.explain(mode="simple")   # look for PartitionFilters
df_pruned.count()

## 7. bucketBy() — Eliminate Join Shuffles

Bucketing physically pre-sorts data into a fixed number of files based on a column hash.
When two tables are bucketed on the **same column with the same bucket count**, joins skip the shuffle entirely.

In [ ]:
(df.write.format("parquet").bucketBy(8, "department").sortBy("department")
      .mode("overwrite").saveAsTable("default.emp_bucketed"))

dept_df = spark.createDataFrame(
    [("Engineering","NYC"),("Marketing","SF"),("HR","NYC"),("Finance","Chicago")],
    ["department","city"]
)
(dept_df.write.format("parquet").bucketBy(8, "department").sortBy("department")
        .mode("overwrite").saveAsTable("default.dept_bucketed"))

# Join with no Exchange (shuffle) in the plan
bj = spark.table("default.emp_bucketed").join(spark.table("default.dept_bucketed"), "department")
bj.explain(mode="simple")   # no Exchange nodes = no shuffle!

## Key Takeaways

| Concept | Use Case |
|---|---|
| `repartition(n)` | Increase parallelism, balance skewed data (triggers shuffle) |
| `repartition(n, col)` | Hash-partition by column before a join |
| `coalesce(n)` | Reduce partitions before writing (no shuffle) |
| `shuffle.partitions` | Tune for dataset size — default 200 is too high for small data |
| Salting | Fix data skew by splitting hot keys across random sub-buckets |
| `partitionBy` on write | Folder-level partitioning — enables partition pruning on reads |
| `bucketBy` | Pre-shuffle data — eliminates join shuffles permanently |